In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import scipy.stats as st
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None) 
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', '{:.3f}'.format) 

sns.set_style("whitegrid")

In [ ]:
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "processed_data" / "Dataset_filtered.csv"
df = pd.read_csv(file_path)

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df = df.drop(labels=['Restaurant Name', 'Address',
       'Locality', 'Locality Verbose', 'Switch to order menu','Rating color', 'Rating text'], axis=1)
df

In [ ]:
df.info()

In [ ]:
num_cols = [col for col in df.columns if (df[col].dtypes=='int64') or (df[col].dtypes=='float64')]
cat_cols = [col for col in df.columns if col not in num_cols]

print(num_cols,'\n',cat_cols)

In [ ]:
# Separating independent and dependent variables
X = df.drop(labels='Aggregate rating',axis=1)
y = df['Aggregate rating']

In [ ]:
X.head()

In [ ]:
y[:5]

In [ ]:
from sklearn.model_selection import train_test_split

rating_bucket = pd.cut(
    y,
    bins=[-float("inf"), 2.5, 3.5, 4.0, float("inf")],
    labels=["low", "average", "good", "excellent"]
)

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=rating_bucket
    )

In [ ]:
print(X_train.shape, y_train.shape)

In [ ]:
print(X_test.shape, y_test.shape)

In [ ]:
# Binary Encoding
binary_mapping = {'No':0, 'Yes':1}

binary_columns = ['Has Table booking', 'Has Online delivery', 'Is delivering now']

for col in binary_columns:
    X_train[col] = X_train[col].replace(binary_mapping)
    X_test[col] = X_test[col].replace(binary_mapping)

X_train.head()

In [ ]:
df_train = pd.concat([X_train,y_train], axis=1)
df_test = pd.concat([X_test,y_test], axis=1)

## Encoding City

In [ ]:
# Target driven encoding for 'City'

train_rated = df_train[df_train['Aggregate rating']>0]

city_avg_rating_train = (train_rated.groupby(['City'])['Aggregate rating']
                        .agg(average_rating='mean')
                        .sort_values(by='average_rating')
                        .reset_index())
city_mapping_train = dict(zip(city_avg_rating_train['City'], city_avg_rating_train['average_rating']))

test_rated = df_test[df_test['Aggregate rating']>0]

city_avg_rating_test = (test_rated.groupby(['City'])['Aggregate rating']
                        .agg(average_rating='mean')
                        .sort_values(by='average_rating')
                        .reset_index())
city_mapping_test = dict(zip(city_avg_rating_test['City'], city_avg_rating_test['average_rating']))

In [ ]:
X_train['City'] = X_train['City'].replace(city_mapping_train)
X_test['City'] = X_test['City'].replace(city_mapping_test)

X_train.head()

## Encoding Currency

In [ ]:
# Target driven encoding for 'Currency'

train_rated = df_train[df_train['Aggregate rating']>0]

currency_avg_rating_train = (train_rated.groupby(['Currency'])['Aggregate rating']
                        .agg(average_rating='mean')
                        .sort_values(by='average_rating')
                        .reset_index())
currency_mapping_train = dict(zip(currency_avg_rating_train['Currency'], currency_avg_rating_train['average_rating']))

test_rated = df_test[df_test['Aggregate rating']>0]

currency_avg_rating_test = (test_rated.groupby(['Currency'])['Aggregate rating']
                        .agg(average_rating='mean')
                        .sort_values(by='average_rating')
                        .reset_index())
currency_mapping_test = dict(zip(currency_avg_rating_test['Currency'], currency_avg_rating_test['average_rating']))

In [ ]:
X_train['Currency'] = X_train['Currency'].replace(currency_mapping_train)
X_test['Currency'] = X_test['Currency'].replace(currency_mapping_test)

X_train.head()

## Encoding Country Code

In [ ]:
# Target driven encoding for 'Country Code'

train_rated = df_train[df_train['Aggregate rating']>0]

country_avg_rating_train = (train_rated.groupby(['Country Code'])['Aggregate rating']
                        .agg(average_rating='mean')
                        .sort_values(by='average_rating')
                        .reset_index())
country_mapping_train = dict(zip(country_avg_rating_train['Country Code'], country_avg_rating_train['average_rating']))

test_rated = df_test[df_test['Aggregate rating']>0]

country_avg_rating_test = (test_rated.groupby(['Country Code'])['Aggregate rating']
                        .agg(average_rating='mean')
                        .sort_values(by='average_rating')
                        .reset_index())
country_mapping_test = dict(zip(country_avg_rating_test['Country Code'], country_avg_rating_test['average_rating']))

In [ ]:
X_train['Country Code'] = X_train['Country Code'].replace(country_mapping_train)
X_test['Country Code'] = X_test['Country Code'].replace(country_mapping_test)

X_train.head()

## Add cuisine count as an extra numeric feature

In [ ]:
X_train["Cuisine_Count"] = X_train["Cuisines"].str.split(", ").apply(len)
X_test["Cuisine_Count"] = X_test["Cuisines"].str.split(", ").apply(len)

X_train.head()

## Encoding cusines according to rating

In [ ]:
# Explode multi-cuisine entries so each row = one cuisine
df_train = (
    df_train.assign(Cuisine=df_train["Cuisines"].str.split(", "))
        .explode("Cuisine")
)
df_train["Cuisine"] = df_train["Cuisine"].str.strip()
df_train.head()

In [ ]:
stats_1 = (
    df_train.groupby("Cuisine")["Aggregate rating"]
            .agg(Avg_Rating="mean")
            .reset_index()
)
cuisine_mapping_train = dict(zip(stats_1['Cuisine'], stats_1['Avg_Rating']))

In [ ]:
df_train['Cuisine'] = df_train['Cuisine'].replace(cuisine_mapping_train)
df_train.head()

In [ ]:
cuisine_rating_train = (
    df_train.groupby("Restaurant ID")['Cuisine']
    .agg(Cuisine_avg_rating='mean')
    .reset_index()
)
cuisine_rating_train

In [ ]:
cuisine_rating_train.loc[cuisine_rating_train['Restaurant ID']==69024, 'Cuisine_avg_rating'].squeeze()

In [ ]:
X_train = pd.merge(X_train, cuisine_rating_train,
                   how='inner',on='Restaurant ID')
X_train.drop('Cuisines', axis=1, inplace=True)
X_train.head()

In [ ]:
# For test data
df_test = (
    df_test.assign(Cuisine=df_test["Cuisines"].str.split(", "))
        .explode("Cuisine")
)
df_test["Cuisine"] = df_test["Cuisine"].str.strip()

stats_2 = (
    df_test.groupby("Cuisine")["Aggregate rating"]
            .agg(Avg_Rating="mean")
            .reset_index()
)
cuisine_mapping_test = dict(zip(stats_2['Cuisine'], stats_2['Avg_Rating']))

df_test['Cuisine'] = df_test['Cuisine'].replace(cuisine_mapping_test)

cuisine_rating_test = (
    df_test.groupby("Restaurant ID")['Cuisine']
    .agg(Cuisine_avg_rating='mean')
    .reset_index()
)


X_test = pd.merge(X_test, cuisine_rating_test,
                   how='inner',on='Restaurant ID')
X_test.drop('Cuisines', axis=1, inplace=True)
X_test.head()

In [ ]:
X_train.drop('Restaurant ID', axis=1, inplace=True)
X_train

In [ ]:
y_train.reset_index(drop=True, inplace=True)
y_train

In [ ]:
X_test.drop('Restaurant ID', axis=1, inplace=True)
X_test

In [ ]:
y_test.reset_index(drop=True, inplace=True)
y_test

In [ ]:
num_cols

In [ ]:
num_cols.remove('Restaurant ID')
num_cols.remove('Country Code')
num_cols.remove('Price range')
num_cols.remove('Aggregate rating')
num_cols

## Getting an overview of the numerical features

In [ ]:
for col in num_cols:
    print(f"________{col}________")
    print(f"Skewness: {df[col].skew().round(3)}")
    print(f"Kurtosis: {st.kurtosis(df[col], fisher=False).round(3)}")
    print("="*20)
    print("\n")

In [ ]:
plt.figure(figsize=(20, 5 * len(num_cols)))

for i, col in enumerate(num_cols, 1):
    plt.subplot(len(num_cols), 1, i)
    sns.histplot(df[col], kde=True)
    plt.title(col)

plt.tight_layout()
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Distribution_Analysis_of_numerical_features.png"
plt.savefig(file_path, dpi=150, bbox_inches="tight")
plt.show()

**Observation:**
- Votes and Average Cost are highly right-skewed → log1p transform required.

# Correlation Analysis

In [ ]:
# Correlation within the input features

corr_value = X_train.corr()
corr_value

In [ ]:
plt.figure(figsize=(15,5))
sns.heatmap(corr_value, annot=True, cmap='coolwarm')
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Inter-correlation_of_input_features.png"
plt.savefig(file_path, dpi=150, bbox_inches="tight");

In [ ]:
# Correlation with target 
target = pd.DataFrame(y_train)
corr_value_1 = X_train.corrwith(target['Aggregate rating']).sort_values(ascending=False)
corr_value_1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

corr_value_1[::-1].plot(
    kind='barh',
    ax=axes[0]
)
axes[0].set_title('Feature Correlation with Target')


sns.heatmap(
    corr_value_1.to_frame(name='Correlation with target'),
    annot=True,
    cmap='coolwarm',
    linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title('Correlation Heatmap')


fig.suptitle('Feature vs Target Correlation Analysis', fontsize=16)

plt.tight_layout()
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "correlation_with_target.png"
plt.savefig(file_path, dpi=150, bbox_inches="tight")
plt.show()

# Outlier Detection

In [ ]:
df.head()

In [ ]:
def detect_outliers(dataset, cols):
    outlier_dict = {}
    for col in cols:
        Q1 = dataset[col].quantile(0.25)
        Q3 = dataset[col].quantile(0.75)
        IQR = Q3 - Q1

        upper_whisker = Q3 + (1.5*IQR)
        lower_whisker = Q1 - (1.5*IQR)

        outliers_list = []
        for val in dataset[col]:
            if (val < lower_whisker) or (val > upper_whisker):
                outliers_list.append(val)

        if len(outliers_list) > 0:
            outliers_list.sort()
            outlier_dict[col] = {'Observation': 'Outliers Detected',
                                'Outliers_count': len(outliers_list),
                                 'Lower_Whisker': lower_whisker,
                                 'Upper_Whisker': upper_whisker,
                                'Detected_outliers': outliers_list}
        else:
            outlier_dict[col] = {'Observation': 'No Outliers Detected'}

    return outlier_dict

In [ ]:
req_cols = ['Average Cost for two','Aggregate rating','Votes']
result = detect_outliers(df, req_cols)
for key, value in result.items():
    print(f"__________{key}__________\n{value}")

## Log transformation:

In [ ]:
cols = ['Average Cost for two','Votes']
for col in cols:
    X_train[col] = np.log1p(X_train[col].clip(lower=0))
    X_test[col] = np.log1p(X_test[col].clip(lower=0))

In [ ]:
X_train

## Normalization

- Among MinMaxScaler, StandardScaler & RobustScaler we are going to use RobustScaler because RobustScaler uses median + IQR, making it immune to those extremes.

In [ ]:
from sklearn.preprocessing import RobustScaler

cols = ['Longitude', 'Latitude', 'Average Cost for two', 'Votes']
scaler = RobustScaler()
X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

In [ ]:
X_train.head()

In [ ]:
X_test.head()

# Final Check

In [ ]:
X_train.head()

In [ ]:
X_train.info()

In [ ]:
X_test.info()

In [ ]:
float_cols = ['City','Currency','Cuisine_avg_rating']
int_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now']

for col in float_cols:
    X_train[col] = X_train[col].astype('float64')
    X_test[col] = X_test[col].astype('float64')

for col in int_cols:
    X_train[col] = X_train[col].astype('int64')
    X_test[col] = X_test[col].astype('int64')

In [ ]:
print(X_train.info())
print()
print(X_test.info())

## Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [ ]:
## Defining Base Models
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "GradientBoost Regressor": GradientBoostingRegressor(),
    "XGBoost Regressor": XGBRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print('          ',list(models.keys())[i],'          ')
    print()
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('-'*35)
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

### After considering all above models we ar going to choose GradientBoost, Random Forest and XGBoost for hyper parameter tunning

# Hyperparameter Tunning with Random Search CV

In [ ]:
#Initialize few parameter for Hyperparamter tuning

gradient_params = {"loss": ['squared_error','huber','absolute_error'],
             "criterion": ['friedman_mse','squared_error','mse'],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500],
              "max_depth": [5, 8, 15, None, 10],
               "learning_rate": [0.1, 0.01, 0.02, 0.03]
            }

rforest_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

xgboost_params = {"learning_rate": [0.1, 0.01],
                  "max_depth": [5, 8, 12, 20, 30],
                  "n_estimators": [100, 200, 300, 500, 1000],
                  "colsample_bytree": [0.5, 0.8, 1, 0.3, 0.4]}

In [ ]:
# Models list for Hyperparameter tuning
randomcv_models = [("GB", GradientBoostingRegressor(), gradient_params),
                   ("RF", RandomForestRegressor(), rforest_params),
                   ("XB", XGBRegressor(), xgboost_params),
                   ]

In [ ]:
# Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=5,
                                   verbose=2,
                                   n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

In [ ]:
# Retraining the models with best parameters
models = {
    "GradientBoost Regressor": GradientBoostingRegressor(n_estimators=500, min_samples_split=15,
                                                        loss='squared_error', learning_rate=0.03, criterion='friedman_mse'),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=500,min_samples_split=2,
                                                     max_features=5,max_depth=15,n_jobs=-1),
    "XGBoost Regressor": XGBRegressor(n_estimators=1000,max_depth=5,learning_rate=0.01,cosample_bytree=1,n_jobs=-1)
    
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print('          ',list(models.keys())[i],'          ')
    print()
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('-'*35)
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

### After considering the above performace we can finalize XGBoost as the model.

In [ ]:
final_model = XGBRegressor(n_estimators=1000,max_depth=5,learning_rate=0.01,cosample_bytree=1,n_jobs=-1)

final_model.fit(X_train, y_train)